In [ ]:
# --- BƯỚC 0: SETUP FOR RTX PRO 6000 BLACKWELL (KAGGLE OFFLINE) ---
import os
import subprocess
import sys
import warnings

warnings.filterwarnings("ignore")

OFFLINE_MODE = True
OFFLINE_DATASET_CANDIDATES = [
    "/kaggle/input/datasets/namthi/offline-kaggle-deps/offline_kaggle_deps",
    "/kaggle/input/datasets/namthi/offline-kaggle-deps",
    "/kaggle/input/offline-kaggle-deps/offline_kaggle_deps",
    "/kaggle/input/offline-kaggle-deps",
]

TORCH_TARGET_VERSION = "2.7.0"

def _resolve_offline_dataset_root(candidates):
    for base in candidates:
        probe_dirs = [base, os.path.join(base, "offline_kaggle_deps"), os.path.join(base, "offline-kaggle-deps")]
        for d in probe_dirs:
            if not os.path.isdir(d):
                continue
            if os.path.isdir(os.path.join(d, "wheels")) and os.path.isfile(os.path.join(d, "requirements-runtime.txt")):
                return d
    return None

OFFLINE_DATASET_DIR = _resolve_offline_dataset_root(OFFLINE_DATASET_CANDIDATES)
OFFLINE_WHEEL_DIR = os.path.join(OFFLINE_DATASET_DIR, "wheels") if OFFLINE_DATASET_DIR else ""
OFFLINE_REQ_FILE = os.path.join(OFFLINE_DATASET_DIR, "requirements-runtime.txt") if OFFLINE_DATASET_DIR else ""

def _run(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)

def _current_py_tag():
    return f"cp{sys.version_info.major}{sys.version_info.minor}"

def _wheel_matches_python_tag(filename):
    low = filename.lower()
    return ("py3-none-any" in low) or ("py3-none-manylinux" in low) or (_current_py_tag() in low)

def _find_wheel(package_name, version_prefix=None, require_python_match=False):
    if not os.path.isdir(OFFLINE_WHEEL_DIR):
        return None
    prefix = f"{package_name.lower()}-"
    cands = []
    for fn in os.listdir(OFFLINE_WHEEL_DIR):
        low = fn.lower()
        if not low.endswith(".whl"):
            continue
        if not low.startswith(prefix):
            continue
        if version_prefix and f"-{version_prefix}" not in low:
            continue
        if require_python_match and (not _wheel_matches_python_tag(fn)):
            continue
        cands.append(fn)
    if not cands:
        return None
    return os.path.join(OFFLINE_WHEEL_DIR, sorted(cands)[-1])

if OFFLINE_MODE and not OFFLINE_DATASET_DIR:
    raise FileNotFoundError(f"Offline dataset not found. Checked: {OFFLINE_DATASET_CANDIDATES}")

print(f">>> Using torch target: {TORCH_TARGET_VERSION}")
print(f">>> Runtime Python tag: {_current_py_tag()}")

torch_was_pinned = False

if OFFLINE_MODE:
    print(">>> OFFLINE MODE: install dependencies from dataset")
    print(f">>> OFFLINE_DATASET_DIR = {OFFLINE_DATASET_DIR}")

    if not os.path.isdir(OFFLINE_WHEEL_DIR):
        raise FileNotFoundError(f"Wheels folder not found: {OFFLINE_WHEEL_DIR}")
    if not os.path.isfile(OFFLINE_REQ_FILE):
        raise FileNotFoundError(f"requirements-runtime.txt not found: {OFFLINE_REQ_FILE}")

    filtered_req = "/kaggle/working/requirements-runtime-filtered.txt"
    skip_prefixes = ("pyarrow", "torch", "torchvision", "torchaudio", "triton", "numpy", "scipy")
    with open(OFFLINE_REQ_FILE, "r", encoding="utf-8") as rf, open(filtered_req, "w", encoding="utf-8") as wf:
        for line in rf:
            s = line.strip().lower()
            if (not s) or s.startswith("#"):
                wf.write(line)
                continue
            normalized = s.replace(" ", "")
            if normalized.startswith(skip_prefixes):
                continue
            wf.write(line)

    print(">>> Installing non-torch deps (filtered, --no-deps)...")
    _run([
        sys.executable, "-m", "pip", "install",
        "--no-index", "--find-links", OFFLINE_WHEEL_DIR,
        "--no-deps",
        "-r", filtered_req,
    ])

    cusparselt_whls = [
        f for f in os.listdir(OFFLINE_WHEEL_DIR)
        if f.endswith(".whl") and "cusparselt" in f.lower()
    ]
    for whl in cusparselt_whls:
        whl_path = os.path.join(OFFLINE_WHEEL_DIR, whl)
        print(f">>> Installing NVIDIA lib: {whl}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", whl_path])

    torch_whl = _find_wheel("torch", TORCH_TARGET_VERSION, require_python_match=True)
    if not torch_whl:
        available = [f for f in os.listdir(OFFLINE_WHEEL_DIR) if f.lower().startswith("torch-") and f.endswith(".whl")]
        print(f"WARN: No torch {TORCH_TARGET_VERSION} wheel found. Available: {available}")
        print("WARN: Keep using runtime torch preinstalled by Kaggle.")
    else:
        print(f">>> Installing torch: {os.path.basename(torch_whl)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", torch_whl])
        torch_was_pinned = True

    tv_whl = _find_wheel("torchvision", require_python_match=True)
    ta_whl = _find_wheel("torchaudio", require_python_match=True)
    if tv_whl:
        print(f">>> Installing torchvision: {os.path.basename(tv_whl)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", tv_whl])
    if ta_whl:
        print(f">>> Installing torchaudio: {os.path.basename(ta_whl)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", ta_whl])

    colpali_whls = [f for f in os.listdir(OFFLINE_WHEEL_DIR) if f.endswith(".whl") and "colpali" in f.lower()]
    if colpali_whls:
        colpali_path = os.path.join(OFFLINE_WHEEL_DIR, sorted(colpali_whls)[0])
        print(f">>> Installing colpali-engine (--no-deps): {os.path.basename(colpali_path)}")
        _run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", OFFLINE_WHEEL_DIR, "--no-deps", colpali_path])
    else:
        raise FileNotFoundError("No colpali-engine wheel found.")

else:
    print(">>> ONLINE MODE")
    _run([sys.executable, "-m", "pip", "uninstall", "-y", "tensorflow", "pyarrow"])
    _run([sys.executable, "-m", "pip", "install", "pyarrow<20.0.0"])
    _run([sys.executable, "-m", "pip", "install", "-U", "git+https://github.com/illuin-tech/colpali"])
    _run([sys.executable, "-m", "pip", "install", "-U", f"torch=={TORCH_TARGET_VERSION}", "transformers", "accelerate", "peft", "bitsandbytes"])
    _run([sys.executable, "-m", "pip", "install", "-U", "torchvision", "torchaudio"])

import ctypes
import glob

nvidia_lib_dirs = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib")
if nvidia_lib_dirs:
    existing_ld = os.environ.get("LD_LIBRARY_PATH", "")
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + (":" + existing_ld if existing_ld else "")
    print(f">>> Added {len(nvidia_lib_dirs)} NVIDIA lib dirs to LD_LIBRARY_PATH")

for pattern in [
    "/usr/local/lib/python*/dist-packages/nvidia/cusparselt/lib/libcusparseLt.so*",
    "/usr/local/lib/python*/dist-packages/nvidia/*/lib/lib*.so*",
]:
    for lib_path in sorted(glob.glob(pattern)):
        try:
            ctypes.CDLL(lib_path, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass

cusparselt_found = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/cusparselt/lib/libcusparseLt.so*")
if cusparselt_found:
    print(f">>> Preloaded libcusparseLt: {cusparselt_found[0]}")
else:
    print("WARN: libcusparseLt.so not found in nvidia packages!")

import torch

print(f">>> torch version: {torch.__version__}")
if torch_was_pinned and (not torch.__version__.startswith(TORCH_TARGET_VERSION)):
    raise RuntimeError(
        f"Expected torch {TORCH_TARGET_VERSION}, but got {torch.__version__}. "
        "Please verify offline wheel pack and restart kernel."
    )
if not torch_was_pinned:
    print(">>> Using runtime torch (wheel pin skipped).")

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    gpu_name = props.name
    vram_gb = props.total_memory / (1024 ** 3)
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    print(f">>> GPU: {gpu_name} | VRAM: {vram_gb:.1f} GB | CC={cc_major}.{cc_minor}")
else:
    raise RuntimeError("CUDA is not available.")

PERF_CFG = {
    "GPU_NAME": gpu_name,
    "VRAM_GB": round(vram_gb, 1),
    "BATCH_SIZE_ENCODE": 12,
    "BATCH_Q_EVAL": 12,
    "DOC_CHUNK_SIZE": 1024,
    "SAVE_EVERY": 900,
}
print(
    ">>> RTX6000 config: "
    f"BATCH_SIZE_ENCODE={PERF_CFG['BATCH_SIZE_ENCODE']}, "
    f"BATCH_Q_EVAL={PERF_CFG['BATCH_Q_EVAL']}, "
    f"DOC_CHUNK_SIZE={PERF_CFG['DOC_CHUNK_SIZE']}"
)

print(">>> Setup dependencies done")

In [ ]:
# --- PATCH: Fix colpali_engine import conflict (KEEP paligemma for ColPali) ---
import sys, types

def _patch_colpali_stub_non_paligemma():
    stale = [k for k in list(sys.modules.keys()) if k.startswith("colpali_engine")]
    for k in stale: del sys.modules[k]
    problem_modules = [
        "colpali_engine.models.gemma3","colpali_engine.models.gemma3.bigemma3","colpali_engine.models.gemma3.colgemma3",
        "colpali_engine.models.modernvbert","colpali_engine.models.modernvbert.bivbert","colpali_engine.models.modernvbert.colvbert",
        "colpali_engine.models.qwen2","colpali_engine.models.qwen2.biqwen2","colpali_engine.models.qwen2.colqwen2",
        "colpali_engine.models.qwen3","colpali_engine.models.qwen3.biqwen3","colpali_engine.models.qwen3.colqwen3",
        "colpali_engine.models.qwen3_5","colpali_engine.models.qwen3_5.biqwen3_5","colpali_engine.models.qwen3_5.colqwen3_5",
        "colpali_engine.models.qwen_omni","colpali_engine.models.qwen_omni.colqwen2_5_omni",
    ]
    for name in problem_modules:
        if name not in sys.modules: sys.modules[name] = types.ModuleType(name)
    stub_map = {
        "colpali_engine.models.gemma3":["BiGemma3","BiGemmaProcessor3","ColGemma3","ColGemmaProcessor3"],
        "colpali_engine.models.modernvbert":["BiModernVBert","BiModernVBertProcessor","ColModernVBert","ColModernVBertProcessor"],
        "colpali_engine.models.qwen2":["BiQwen2","BiQwen2Processor","ColQwen2","ColQwen2Processor"],
        "colpali_engine.models.qwen3":["BiQwen3","BiQwen3Processor","ColQwen3","ColQwen3Processor"],
        "colpali_engine.models.qwen3_5":["BiQwen3_5","BiQwen3_5Processor","ColQwen3_5","ColQwen3_5Processor"],
        "colpali_engine.models.qwen_omni":["ColQwen2_5Omni","ColQwen2_5OmniProcessor"],
    }
    for mod_name, cls_list in stub_map.items():
        mod = sys.modules[mod_name]
        for cls_name in cls_list: setattr(mod, cls_name, type(cls_name, (), {}))

_patch_colpali_stub_non_paligemma()
from colpali_engine.models.paligemma import ColPali, ColPaliProcessor
print(f"ColPali ready: {ColPali}")


In [ ]:
# --- BƯỚC 2: LOAD MODEL (ColPali v1.3) ---
print(">>> BƯỚC 2: Loading ColPali v1.3...")

import os
import gc
import torch
import transformers
from peft import PeftModel, PeftConfig

try:
    from transformers import PaliGemmaForConditionalGeneration
    if not hasattr(PaliGemmaForConditionalGeneration, "model"):
        PaliGemmaForConditionalGeneration.model = property(lambda self: self)
        print(">>> Applied class-level compat patch: PaliGemmaForConditionalGeneration.model -> self")
except Exception as e:
    print(f">>> WARN: compat patch skipped: {e}")

from colpali_engine.models.paligemma import ColPali, ColPaliProcessor

gc.collect()
torch.cuda.empty_cache()

print(f">>> transformers version: {transformers.__version__}")

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    major, minor = torch.cuda.get_device_capability(0)
    gpu_name = torch.cuda.get_device_name(0)
    if major >= 8:
        RUNTIME_DTYPE = torch.bfloat16
    else:
        RUNTIME_DTYPE = torch.float16
    print(f">>> GPU: {gpu_name} | CC={major}.{minor} | Runtime dtype={RUNTIME_DTYPE}")
else:
    RUNTIME_DTYPE = torch.float32

def _first_existing_path(candidates):
    for p in candidates:
        if os.path.isdir(p):
            return p
    return None

COLPALI_MODEL_CANDIDATES = [
    "/kaggle/input/datasets/namthi/colpali-v1-3",
    "/kaggle/input/datasets/namthi/colpali-v1-3/colpali-v1.3",
    "/kaggle/input/colpali-v1-3",
    "/kaggle/input/colpali-v1.3",
]

BASE_MODEL_CANDIDATES = [
    "/kaggle/input/datasets/namthi/paligemma-3b-pt-448",
    "/kaggle/input/paligemma-3b-pt-448",
]

model_dir = _first_existing_path(COLPALI_MODEL_CANDIDATES)
base_dir = _first_existing_path(BASE_MODEL_CANDIDATES)

if not model_dir:
    raise FileNotFoundError("Khong tim thay adapter model colpali-v1-3")

has_full_weight = any(os.path.exists(os.path.join(model_dir, f))
                     for f in ["model.safetensors", "pytorch_model.bin"]) or \
                  any(f.startswith("model-") for f in os.listdir(model_dir))

if has_full_weight:
    print(">>> Detected full weights. Loading directly...")
    model = ColPali.from_pretrained(
        model_dir,
        torch_dtype=RUNTIME_DTYPE,
        device_map="auto" if device == "cuda" else "cpu",
        attn_implementation="eager",
        local_files_only=True,
    )
    processor = ColPaliProcessor.from_pretrained(model_dir, use_fast=True)
else:
    print(">>> Detected adapter-style. Loading Base + Adapter...")
    if not base_dir:
        raise FileNotFoundError("Khong tim thay base model paligemma-3b-pt-448 để nạp adapter")

    base_model = ColPali.from_pretrained(
        base_dir,
        torch_dtype=RUNTIME_DTYPE,
        device_map="auto" if device == "cuda" else "cpu",
        attn_implementation="eager",
        local_files_only=True,
    )

    model = PeftModel.from_pretrained(
        base_model,
        model_dir,
        local_files_only=True,
    )

    print(">>> Merging weights...")
    model = model.merge_and_unload()

    processor = ColPaliProcessor.from_pretrained(model_dir, use_fast=True)

model = model.eval()

try:
    if not hasattr(model, "model"):
        model.model = model
        print(">>> Applied runtime compat patch: model.model -> model")
except Exception as e:
    print(f">>> Patch warning: {e}")

# Alias theo encode notebook + giữ tương thích với các method hiện tại
query_model = model
query_processor = processor

print(">>> [SUCCESS] Model & Processor ready (ColPali v1.3)")

In [ ]:
# --- BƯỚC 2.5: LOAD PRE-ENCODED INDEX + BUILD QA PAIRS FROM VIDORE V3 ---
# Dataset structure (BEIR format):
#   {domain}/{domain}/corpus/*.parquet   — corpus_id, image, doc_id, markdown, page_number_in_doc
#   {domain}/{domain}/queries/*.parquet  — query_id, query, ...
#   {domain}/{domain}/qrels/*.parquet    — query_id, corpus_id, score
print(">>> Loading pre-encoded ColPali ViDoRe index + queries...")

import os
import pickle
import re
import torch
import pandas as pd
import numpy as np
from pathlib import Path
import glob

# ============================================================
# 1) Load encoded index
# ============================================================
# Optional: set this manually when you upload .pkl as a Kaggle Dataset in a separate session.
# Example: INDEX_PATH_OVERRIDE = '/kaggle/input/my-encoded-index/colpali13_page_index_vidore_v3_pagelevel.pkl'
INDEX_PATH_OVERRIDE = globals().get('INDEX_PATH_OVERRIDE', None)

INDEX_CANDIDATES = [
    INDEX_PATH_OVERRIDE,
    '/kaggle/working/colpali13_page_index_vidore_v3_pagelevel.pkl',
    '/kaggle/input/datasets/namthi/vidore-encoded/colpali13_page_index_vidore_v3_pagelevel.pkl',
]
# Auto-discover uploaded index files in Kaggle inputs
INDEX_CANDIDATES.extend(sorted(glob.glob('/kaggle/input/**/colpali13_page_index_vidore_v3_pagelevel.pkl', recursive=True)))
INDEX_PATH = next((p for p in INDEX_CANDIDATES if p and os.path.exists(p)), None)
if not INDEX_PATH:
    raise FileNotFoundError(f"Index not found. Checked: {INDEX_CANDIDATES}")
print(f">>> Using INDEX_PATH: {INDEX_PATH}")

with open(INDEX_PATH, "rb") as f:
    payload = pickle.load(f)

if isinstance(payload, dict):
    fused_index = payload.get("fused_index", payload.get("embeddings", []))
    page_keys = payload.get("page_keys", [])
    metadata_list = payload.get("metadata", [])
    print(f">>> Model: {payload.get('model_name', 'N/A')}, Dataset: {payload.get('dataset', 'N/A')}")
    print(f">>> num_pages: {payload.get('num_pages', 'N/A')}")
    model_name = str(payload.get('model_name', ''))
    dataset_name = str(payload.get('dataset', ''))
    index_level = str(payload.get('index_level', ''))
    if model_name and ("colpali-v1.3" not in model_name.lower()):
        raise ValueError(f"Index model mismatch: {model_name}. Expected ColPali v1.3 index from encode notebook.")
    if dataset_name and ("vidore-v3" not in dataset_name.lower()):
        raise ValueError(f"Index dataset mismatch: {dataset_name}. Expected vidore-v3.")
    if index_level and (index_level.lower() != "page"):
        raise ValueError(f"Index level mismatch: {index_level}. Expected page-level index.")
else:
    fused_index = payload
    page_keys = []
    metadata_list = []

all_page_embeddings = []
for emb in fused_index:
    if isinstance(emb, torch.Tensor):
        all_page_embeddings.append(emb.float().cpu().numpy())
    elif isinstance(emb, np.ndarray):
        all_page_embeddings.append(emb.astype(np.float32))
    else:
        all_page_embeddings.append(np.array(emb, dtype=np.float32))

print(f">>> Loaded {len(all_page_embeddings)} page embeddings")
if all_page_embeddings:
    print(f">>> Embedding shape sample: {all_page_embeddings[0].shape}")


def _normalize_corpus_id(v):
    if v is None:
        return None
    if isinstance(v, float) and np.isnan(v):
        return None
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        if np.isnan(v):
            return None
        return str(int(v)) if float(v).is_integer() else str(v)

    s = str(v).strip()
    if not s:
        return None
    if re.fullmatch(r"\d+\.0+", s):
        return s.split(".")[0]
    return s


# IMPORTANT: use domain-aware corpus_id lookup to avoid cross-domain id collision
# Also keep one-to-many mapping: one corpus_id can map to multiple page rows.
meta_df = pd.DataFrame(metadata_list) if metadata_list else pd.DataFrame()
domain_cid_to_indices = {}
global_cid_to_indices = {}
global_collision_count = 0
domain_duplicate_pages = 0
global_duplicate_pages = 0

if not meta_df.empty and "corpus_id" in meta_df.columns:
    if "domain" not in meta_df.columns:
        meta_df["domain"] = "unknown"

    for idx, row in meta_df.iterrows():
        domain = str(row.get("domain", "unknown")).strip() or "unknown"
        cid = _normalize_corpus_id(row.get("corpus_id"))
        if cid is None:
            continue

        key = (domain, cid)
        if key not in domain_cid_to_indices:
            domain_cid_to_indices[key] = []
        else:
            domain_duplicate_pages += 1
        domain_cid_to_indices[key].append(idx)

        # Track global collisions/duplicates across domains
        if cid not in global_cid_to_indices:
            global_cid_to_indices[cid] = []
        elif global_cid_to_indices[cid] and global_cid_to_indices[cid][-1] != idx:
            global_collision_count += 1
            global_duplicate_pages += 1
        global_cid_to_indices[cid].append(idx)

    print(f">>> domain+corpus_id unique keys: {len(domain_cid_to_indices)}")
    print(f">>> domain duplicate pages (same domain+corpus_id): {domain_duplicate_pages}")
    print(f">>> global corpus_id collisions detected: {global_collision_count}")
    print(f">>> global duplicate pages (same corpus_id across/all domains): {global_duplicate_pages}")
    print(f">>> domains in metadata: {sorted(meta_df['domain'].astype(str).unique().tolist())}")
else:
    # fallback parse page_keys: domain__doc__pX__corpus_id
    for idx, pk in enumerate(page_keys):
        parts = str(pk).split("__")
        if len(parts) >= 4:
            domain = parts[0]
            cid = _normalize_corpus_id(parts[-1])
            if cid is None:
                continue

            key = (domain, cid)
            if key not in domain_cid_to_indices:
                domain_cid_to_indices[key] = []
            else:
                domain_duplicate_pages += 1
            domain_cid_to_indices[key].append(idx)

            if cid not in global_cid_to_indices:
                global_cid_to_indices[cid] = []
            elif global_cid_to_indices[cid] and global_cid_to_indices[cid][-1] != idx:
                global_collision_count += 1
                global_duplicate_pages += 1
            global_cid_to_indices[cid].append(idx)

    print(f">>> domain+corpus_id unique keys (from page_keys): {len(domain_cid_to_indices)}")
    print(f">>> domain duplicate pages (from page_keys): {domain_duplicate_pages}")
    print(f">>> global corpus_id collisions detected: {global_collision_count}")
    print(f">>> global duplicate pages (from page_keys): {global_duplicate_pages}")

if not domain_cid_to_indices:
    raise ValueError("Could not build domain-aware corpus_id lookup from index")

# ============================================================
# 2) Load queries/qrels and build QA pairs
# ============================================================
VIDORE_DATASET_ROOT = "/kaggle/input/datasets/namthi/vidore-v3"
VIDORE_DOMAINS = [
    "vidore_v3_computer_science",
    "vidore_v3_energy",
    "vidore_v3_finance_en",
    "vidore_v3_hr",
    "vidore_v3_industrial",
    "vidore_v3_pharmaceuticals",
    "vidore_v3_physics",
]

root = Path(VIDORE_DATASET_ROOT)
if not root.exists():
    raise FileNotFoundError(f"Dataset root not found: {VIDORE_DATASET_ROOT}")

domain_dirs = [root / d for d in VIDORE_DOMAINS if (root / d).is_dir()]
if not domain_dirs:
    domain_dirs = [p for p in sorted(root.iterdir()) if p.is_dir()]

print(f">>> Domains from dataset: {[p.name for p in domain_dirs]}")

qa_pairs = []
match_stats = {
    "matched_domain_key": 0,
    "matched_pages_domain_key": 0,
    "matched_global_fallback": 0,
    "matched_pages_global_fallback": 0,
    "missed": 0,
    "no_qrels": 0,
    "no_queries": 0,
}
missed_samples = []

for domain_dir in domain_dirs:
    domain = domain_dir.name

    qrel_files = sorted(domain_dir.rglob("qrels/*.parquet"))
    if not qrel_files:
        print(f"  WARN: No qrels in {domain}")
        match_stats["no_qrels"] += 1
        continue

    qrels_chunks = []
    for fp in qrel_files:
        try:
            qrels_chunks.append(pd.read_parquet(fp))
        except Exception as e:
            print(f"  Skip qrels {fp}: {e}")
    if not qrels_chunks:
        continue

    qrels_df = pd.concat(qrels_chunks, ignore_index=True)
    if "query_id" not in qrels_df.columns or "corpus_id" not in qrels_df.columns:
        print(f"  WARN: qrels missing required columns in {domain}")
        continue

    # Keep only positive labels if score column exists
    if "score" in qrels_df.columns:
        qrels_df = qrels_df[qrels_df["score"].fillna(0) > 0].copy()

    qrels_df["corpus_id_norm"] = qrels_df["corpus_id"].map(_normalize_corpus_id)
    qrels_df = qrels_df.dropna(subset=["query_id", "corpus_id_norm"]).copy()

    query_files = sorted(domain_dir.rglob("queries/*.parquet"))
    if not query_files:
        print(f"  WARN: No queries in {domain}")
        match_stats["no_queries"] += 1
        continue

    query_chunks = []
    for fp in query_files:
        try:
            query_chunks.append(pd.read_parquet(fp))
        except Exception as e:
            print(f"  Skip queries {fp}: {e}")
    if not query_chunks:
        continue

    queries_df = pd.concat(query_chunks, ignore_index=True)
    if "query_id" not in queries_df.columns or "query" not in queries_df.columns:
        print(f"  WARN: queries missing required columns in {domain}")
        continue

    merged = queries_df[["query_id", "query"]].merge(
        qrels_df[["query_id", "corpus_id_norm"]],
        on="query_id",
        how="inner",
    )

    print(f"  {domain}: qrels_pos={len(qrels_df)}, queries={len(queries_df)}, merged={len(merged)}")

    for qid, grp in merged.groupby("query_id"):
        query_text = str(grp.iloc[0]["query"]).strip()
        if len(query_text) < 3:
            continue

        gt_indices = []
        for _, row in grp.iterrows():
            cid = row["corpus_id_norm"]

            idx_list = domain_cid_to_indices.get((domain, cid))
            if idx_list:
                gt_indices.extend(idx_list)
                match_stats["matched_domain_key"] += 1
                match_stats["matched_pages_domain_key"] += len(idx_list)
                continue

            # fallback if index metadata lacks domain or naming mismatch
            idx_list = global_cid_to_indices.get(cid)
            if idx_list:
                gt_indices.extend(idx_list)
                match_stats["matched_global_fallback"] += 1
                match_stats["matched_pages_global_fallback"] += len(idx_list)
                continue

            match_stats["missed"] += 1
            if len(missed_samples) < 20:
                missed_samples.append({"domain": domain, "query_id": str(qid), "corpus_id": str(cid)})

        if gt_indices:
            qa_pairs.append({
                "question": query_text,
                "gt_embed_indices": sorted(set(gt_indices)),
                "doc_name": domain,
                "domain": domain,
            })

print(f"\n>>> Total QA pairs: {len(qa_pairs)}")
print(f">>> Match stats: {match_stats}")
if qa_pairs:
    gt_sizes = np.array([len(x["gt_embed_indices"]) for x in qa_pairs], dtype=np.int32)
    print(
        f">>> GT pages/query stats: mean={gt_sizes.mean():.2f}, median={np.median(gt_sizes):.1f}, p95={np.percentile(gt_sizes, 95):.1f}, max={gt_sizes.max()}"
    )
    print(
        f">>> Matched pages total: domain={match_stats['matched_pages_domain_key']}, global_fallback={match_stats['matched_pages_global_fallback']}"
    )

num_total_links = (
    match_stats["matched_domain_key"]
    + match_stats["matched_global_fallback"]
    + match_stats["missed"]
)
coverage = (
    (match_stats["matched_domain_key"] + match_stats["matched_global_fallback"]) / num_total_links
    if num_total_links > 0 else 0.0
)

print(f">>> corpus_id overall coverage: {coverage*100:.2f}%")
if num_total_links > 0:
    print(
        f">>> domain-key matched: {match_stats['matched_domain_key']}/{num_total_links}, "
        f"global-fallback: {match_stats['matched_global_fallback']}/{num_total_links}, "
        f"missed: {match_stats['missed']}/{num_total_links}"
    )

if missed_samples:
    print(">>> Sample missed corpus_id (first 20):")
    for x in missed_samples:
        print(f"   - [{x['domain']}] qid={x['query_id']} cid={x['corpus_id']}")

if qa_pairs:
    per_domain = {}
    for x in qa_pairs:
        per_domain[x["domain"]] = per_domain.get(x["domain"], 0) + 1
    print(">>> QA pairs per domain:")
    for d, c in sorted(per_domain.items()):
        print(f"   - {d}: {c}")

if not qa_pairs:
    raise ValueError("No QA pairs found. Check qrels/corpus_id mapping.")

# Hard guard: if this trips, index file is likely not aligned with current qrels split
if coverage < 0.95:
    raise ValueError(
        f"Low corpus_id coverage ({coverage*100:.2f}%). Encoded index may not match current ViDoRe split/domain set."
    )

In [ ]:
# ==============================================================================
# Shared utilities, config, and all helper functions
# Adapted from method base.ipynb for ColPali + ViDoRe
# ==============================================================================
import os, gc, time, pickle, json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.notebook import tqdm

WORKING_DIR = "/kaggle/working"

# ---- CONFIG ----
TOPK_RATIOS        = [round(i*0.1,1) for i in range(1,10)]  # 0.1..0.9
KMEANS_ITERS       = 10
N_RANDOM_SEEDS     = 5
N_LAST_LAYERS_LIST = [4, 8, 16, 32]
NORMALIZE_MODES    = ['pre', 'post']
ATTN_N_LAYERS_LIST = [1, 2, 4, 8]
SVD_RANK_REMOVE    = 1
TEMPERATURE_OURS   = 0.5

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Keep query preprocessing identical to encode notebook text branch.
QUERY_MAX_LENGTH = 512
QUERY_SUFFIX = ""

def build_query_inputs(question, processor, device):
    return processor.process_queries([question], max_length=QUERY_MAX_LENGTH, suffix=QUERY_SUFFIX).to(device)

# ---- Doc matrix builder ----
def build_doc_matrix(embeddings, device):
    arrays = [torch.from_numpy(e).float() if isinstance(e, np.ndarray) else e.float() for e in embeddings]
    max_len = max(a.shape[0] for a in arrays); D = arrays[0].shape[1]; n = len(arrays)
    mat = torch.zeros(n, max_len, D, dtype=torch.float32)
    mask = torch.zeros(n, max_len, dtype=torch.bool)
    for i, a in enumerate(arrays):
        L = a.shape[0]
        mat[i,:L] = F.normalize(a, dim=-1)
        mask[i,:L] = True
    return mat.to(device), mask.to(device)

@torch.no_grad()
def fast_maxsim(q_norm, doc_matrix, doc_mask):
    sim = torch.einsum('qd,nld->qnl', q_norm, doc_matrix)
    sim.masked_fill_(~doc_mask.unsqueeze(0), float('-inf'))
    return sim.max(dim=-1).values

def forward_query_embeddings(model, q_inputs):
    """
    Robust query forward compatible with multiple transformers/colpali output formats.
    Returns normalized token embeddings for tokens where attention_mask=1.
    """
    with torch.no_grad():
        outputs = model(**q_inputs, return_dict=True)

    if hasattr(outputs, "last_hidden_state") and outputs.last_hidden_state is not None:
        proj = outputs.last_hidden_state
    elif isinstance(outputs, torch.Tensor):
        proj = outputs
    elif isinstance(outputs, (tuple, list)) and len(outputs) > 0:
        proj = outputs[0]
    else:
        raise ValueError("Could not extract query embeddings from model output")

    if proj.dim() != 3:
        raise ValueError(f"Unexpected query embedding shape: {tuple(proj.shape)}")

    attn_mask = q_inputs['attention_mask'][0] > 0
    q_emb = proj[0][attn_mask].float()
    return F.normalize(q_emb, dim=-1)

# ---- Metrics ----
def compute_ndcg(ranked, gt, k):
    dcg = sum(1.0/np.log2(r+2) for r,i in enumerate(ranked[:k]) if i in gt)
    idcg = sum(1.0/np.log2(r+2) for r in range(min(len(gt),k)))
    return dcg/idcg if idcg>0 else 0.0

def first_hit(top_k, gt):
    for r,i in enumerate(top_k):
        if i in gt: return r+1
    return -1

def hit_metrics(top10, gt):
    h = first_hit(top10, gt)
    return {'r1':int(h!=-1 and h<=1),'r5':int(h!=-1 and h<=5),'r10':int(h!=-1 and h<=10),
            'n1':float(compute_ndcg(top10,gt,1)),'n5':float(compute_ndcg(top10,gt,5)),'n10':float(compute_ndcg(top10,gt,10))}

def _init_metric():
    return {'r1':0,'r5':0,'r10':0,'n1':0.0,'n5':0.0,'n10':0.0,'count':0}

def _add_metric(dst, src):
    dst['r1']+=int(src['r1']); dst['r5']+=int(src['r5']); dst['r10']+=int(src['r10'])
    dst['n1']+=float(src['n1']); dst['n5']+=float(src['n5']); dst['n10']+=float(src['n10'])
    dst['count']+=1

def _ensure(store, key):
    if key not in store: store[key] = _init_metric()
    return store[key]

def record(all_metrics, all_domain_metrics, key, m, domain):
    _add_metric(_ensure(all_metrics, key), m)
    if domain not in all_domain_metrics: all_domain_metrics[domain] = {}
    _add_metric(_ensure(all_domain_metrics[domain], key), m)

def print_summary(all_metrics, all_domain_metrics, method_keys, title=""):
    if title: print(f"\n{'='*60}\n{title}\n{'='*60}")
    print(f"{'Method':<35} {'R@1':>7} {'R@5':>7} {'R@10':>7} {'nDCG@10':>9}")
    print("-"*65)
    for key in method_keys:
        if key not in all_metrics: continue
        m = all_metrics[key]; cnt = m['count'] or 1
        print(f"{key:<35} {m['r1']/cnt*100:6.2f}%  {m['r5']/cnt*100:6.2f}%  {m['r10']/cnt*100:6.2f}%  {m['n10']/cnt:8.4f}")

# ---- Latency Tracker ----
class LatencyTracker:
    def __init__(self, name):
        self.name = name; self.ratio_ms = {}
    def add_ratio(self, ratio, ms):
        if ratio not in self.ratio_ms: self.ratio_ms[ratio] = []
        self.ratio_ms[ratio].append(ms)
    def report(self):
        if not self.ratio_ms: print(f"[{self.name}] No latency data."); return
        print(f"\n{'='*70}\nLatency — {self.name}\n{'='*70}")
        print(f"  {'Ratio':<10} {'n':>6} {'avg ms':>10} {'p50 ms':>10} {'p95 ms':>10}")
        for ratio in sorted(self.ratio_ms):
            vals = self.ratio_ms[ratio]; n = len(vals)
            print(f"  {ratio:<10.0%} {n:>6} {np.mean(vals):>10.2f} {np.percentile(vals,50):>10.2f} {np.percentile(vals,95):>10.2f}")
    def to_dict(self):
        rows = []
        for ratio in sorted(self.ratio_ms):
            vals = self.ratio_ms[ratio]; n = len(vals)
            rows.append({'method':self.name,'ratio':ratio,'n_queries':n,
                         'avg_score_ms':round(np.mean(vals),3),'p50_score_ms':round(np.percentile(vals,50),3),
                         'p95_score_ms':round(np.percentile(vals,95),3)})
        return rows

# ---- SVD / Importance helpers ----
def remove_sink_components_batch(attn_heads, k):
    try:
        U,S,Vh = torch.linalg.svd(attn_heads, full_matrices=False)
        k = min(k, S.shape[-1])
        sink = (U[...,:k]*S[:,:k].unsqueeze(1))@Vh[:,:k,:]
        return (attn_heads - sink).clamp(min=0.0)
    except: return attn_heads

def compute_svd_importance_softplus(attentions, content_mask, layer_weights, k):
    dev = content_mask.device; B,S = content_mask.shape
    importance = torch.zeros(B,S,device=dev)
    for i,attn in enumerate(attentions):
        attn = attn.float().to(dev); B_,H,Sq,Sk = attn.shape
        cleaned = remove_sink_components_batch(attn.view(B_*H,Sq,Sk),k).view(B_,H,Sq,Sk)
        layer_imp = cleaned.sum(dim=2).mean(dim=1)*content_mask
        layer_imp = layer_imp/layer_imp.max(dim=-1,keepdim=True).values.clamp(min=1e-8)
        importance += layer_weights[i]*layer_imp
    importance = F.softplus(importance*content_mask/TEMPERATURE_OURS)*content_mask
    return importance

def build_content_mask(inputs, processor):
    attn_mask = inputs["attention_mask"]
    input_ids = inputs.get("input_ids",None)
    if input_ids is None: return attn_mask.float()
    tok = getattr(processor,'tokenizer',processor)
    special_ids = set()
    for attr in ['pad_token_id','bos_token_id','eos_token_id','unk_token_id','sep_token_id','cls_token_id']:
        tid = getattr(tok,attr,None)
        if tid is not None: special_ids.add(int(tid))
    if hasattr(tok,'added_tokens_encoder'):
        for _,tid in tok.added_tokens_encoder.items(): special_ids.add(int(tid))
    if not special_ids: return attn_mask.float()
    st = torch.tensor(list(special_ids),device=input_ids.device)
    is_special = (input_ids.unsqueeze(-1)==st).any(dim=-1)
    return attn_mask.float()*(~is_special).float()

# Alias for methods (method base uses build_content_mask_qwen)
build_content_mask_qwen = build_content_mask

def build_cluster_pool_vectors_ours(q_norm_kept, q_norm_disc, imp_kept, imp_disc, normalize_mode='pre'):
    P = q_norm_disc.shape[0]
    if P == 0:
        D = q_norm_kept.shape[1]
        return torch.zeros(0,D,device=q_norm_kept.device), torch.zeros(0,device=q_norm_kept.device)
    sim_assign = torch.mm(q_norm_disc, q_norm_kept.t())
    cluster_ids = sim_assign.argmax(dim=-1)
    pooled_vecs, pooled_weights = [], []
    for c in cluster_ids.unique():
        members = (cluster_ids==c).nonzero(as_tuple=True)[0]
        w = imp_disc[members]; w_sum = w.sum().clamp(min=1e-8); w_norm = w/w_sum
        pool_vec = (q_norm_disc[members]*w_norm.unsqueeze(-1)).sum(dim=0)
        if normalize_mode == 'pre':
            pool_vec = F.normalize(pool_vec.unsqueeze(0),dim=-1).squeeze(0)
        pooled_vecs.append(pool_vec); pooled_weights.append(w_sum)
    return torch.stack(pooled_vecs), torch.stack(pooled_weights)

# ---- Spherical KMeans ----
def spherical_kmeans(X, K, n_iters=KMEANS_ITERS):
    N,D = X.shape; K = min(K,N)
    if K >= N: return X.clone()
    perm = torch.randperm(N,device=X.device)
    centroids = X[perm[:K]].clone()
    for _ in range(n_iters):
        sim = torch.mm(X,centroids.t()); cluster_ids = sim.argmax(dim=1)
        new_c = torch.zeros_like(centroids)
        for c in range(K):
            members = (cluster_ids==c).nonzero(as_tuple=True)[0]
            new_c[c] = X[members].mean(dim=0) if members.numel()>0 else centroids[c]
        centroids = F.normalize(new_c,dim=-1)
    return centroids

# ---- Random pruning ----
def random_prune_topk(q_norm, content_idx, doc_matrix, doc_mask, topk_ratios, n_seeds=N_RANDOM_SEEDS):
    M = q_norm.shape[0]; n_docs = doc_matrix.shape[0]; dev = q_norm.device
    if M == 0:
        zero = torch.zeros(n_docs,device=dev)
        return {r:zero for r in topk_ratios}, {r:0.0 for r in topk_ratios}
    results, timing = {}, {}
    for ratio in topk_ratios:
        k = min(max(1,round(M*ratio)),M)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        if k == M:
            scores = fast_maxsim(q_norm,doc_matrix,doc_mask).sum(dim=0)
        else:
            acc = torch.zeros(n_docs,device=dev,dtype=torch.float32)
            for _ in range(n_seeds):
                perm = torch.randperm(M,device=dev)[:k]
                acc += fast_maxsim(q_norm[perm],doc_matrix,doc_mask).sum(dim=0)
            scores = acc/n_seeds
        torch.cuda.synchronize(); timing[ratio] = (time.perf_counter()-t0)*1000.0
        results[ratio] = scores
    return results, timing

# ---- Encode query live (with attentions for ColPali) ----
def encode_query_live(question, processor, model, device):
    q_inputs = build_query_inputs(question, processor, device)
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        outputs = model(**q_inputs, output_attentions=True, return_dict=True)
    if hasattr(outputs, "last_hidden_state"):
        proj = outputs.last_hidden_state
    elif isinstance(outputs, torch.Tensor):
        proj = outputs
    else:
        proj = outputs[0] if isinstance(outputs, tuple) else outputs
    proj = proj / (proj.norm(dim=-1,keepdim=True)+1e-8)
    proj = proj * q_inputs["attention_mask"].unsqueeze(-1).float()
    torch.cuda.synchronize()
    encode_ms = (time.perf_counter()-t0)*1000.0
    return proj, outputs, q_inputs, encode_ms

# ---- Method key builders ----
def make_ablation_keys_ours():
    keys = []
    for n in N_LAST_LAYERS_LIST:
        for mode in NORMALIZE_MODES:
            for r in TOPK_RATIOS:
                tag = f"L{n}_norm{mode}_r{int(r*100)}"
                keys.append(f"{tag}_trad"); keys.append(f"{tag}_weighted")
    return keys

ABLATION_KEYS_OURS = make_ablation_keys_ours()

def save_summary_csv(metrics, domain_metrics, method_keys, prefix):
    rows = []
    for key in method_keys:
        if key not in metrics: continue
        m = metrics[key]; cnt = m['count'] or 1
        rows.append({'method':key,'r@1':round(m['r1']/cnt*100,4),'r@5':round(m['r5']/cnt*100,4),
                     'r@10':round(m['r10']/cnt*100,4),'ndcg@1':round(m['n1']/cnt,6),
                     'ndcg@5':round(m['n5']/cnt,6),'ndcg@10':round(m['n10']/cnt,6),'count':cnt})
    pd.DataFrame(rows).to_csv(os.path.join(WORKING_DIR,f"{prefix}_summary.csv"),index=False)
    dom_rows = []
    for domain in sorted(domain_metrics):
        dm = domain_metrics[domain]; row = {'domain':domain}
        for key in method_keys:
            m_ = dm.get(key,_init_metric()); cnt_ = m_['count'] or 1
            row[f'{key}_ndcg10'] = round(m_['n10']/cnt_,6); row[f'{key}_r10'] = round(m_['r10']/cnt_*100,4)
        dom_rows.append(row)
    pd.DataFrame(dom_rows).to_csv(os.path.join(WORKING_DIR,f"{prefix}_domain.csv"),index=False)
    print(f"✅ Saved: {prefix}_summary.csv, {prefix}_domain.csv")

print(">>> Utility functions ready")
print(f">>> TOPK_RATIOS={TOPK_RATIOS}")
print(f">>> N_LAST_LAYERS_LIST={N_LAST_LAYERS_LIST}")


In [ ]:
# ==============================================================================
# METHOD 1 (SMALL SAMPLE) — Traditional MaxSim quick check
# ==============================================================================
print(">>> METHOD 1 (SMALL SAMPLE): Traditional MaxSim")

# ---- Quick-run config ----
SMALL_SAMPLE_SIZE = 200      # đổi thành 100/300 tùy tốc độ muốn test
SMALL_SAMPLE_SEED = 42
SMALL_SAMPLE_STRATEGY = "random"  # "random" | "head"

if "qa_pairs" not in globals() or "all_page_embeddings" not in globals():
    raise RuntimeError("Thiếu qa_pairs hoặc all_page_embeddings. Hãy chạy Cell 4 trước.")

if "query_model" not in globals() or "query_processor" not in globals():
    raise RuntimeError("Thiếu query_model/query_processor. Hãy chạy Cell 3 trước.")

n_total = len(qa_pairs)
if n_total == 0:
    raise ValueError("qa_pairs rỗng")

n_take = min(SMALL_SAMPLE_SIZE, n_total)
if SMALL_SAMPLE_STRATEGY == "head":
    sample_indices = list(range(n_take))
else:
    rng = np.random.default_rng(SMALL_SAMPLE_SEED)
    sample_indices = rng.choice(n_total, size=n_take, replace=False).tolist()

sample_qa_pairs = [qa_pairs[i] for i in sample_indices]
print(f">>> Sample queries: {len(sample_qa_pairs)}/{n_total} | strategy={SMALL_SAMPLE_STRATEGY}")

print(f"Building doc matrix from {len(all_page_embeddings)} page embeddings...")
doc_matrix, doc_mask = build_doc_matrix(all_page_embeddings, device)
n_docs = doc_matrix.shape[0]
print(f"Doc matrix shape: {doc_matrix.shape}")

sample_metrics = {}
sample_domain_metrics = {}
sample_latency = LatencyTracker("Traditional MaxSim (sample)")

pbar = tqdm(enumerate(sample_qa_pairs), total=len(sample_qa_pairs), desc="Traditional MaxSim (sample)")
for q_idx, item in pbar:
    question = item['question']
    gt_set = set(item['gt_embed_indices'])
    domain = item['domain']

    q_inputs = build_query_inputs(question, query_processor, device)
    with torch.no_grad():
        q_proj = query_model(**q_inputs)

    if hasattr(q_proj, "last_hidden_state") and q_proj.last_hidden_state is not None:
        q_proj = q_proj.last_hidden_state
    elif isinstance(q_proj, (tuple, list)) and len(q_proj) > 0:
        q_proj = q_proj[0]

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx = torch.where(attn_mask > 0)[0]
    q_emb = q_proj[0][trad_idx].float()
    q_norm = F.normalize(q_emb, dim=-1)

    torch.cuda.synchronize(); t0 = time.perf_counter()
    M = fast_maxsim(q_norm, doc_matrix, doc_mask)
    scores = M.sum(dim=0)
    top10 = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()
    torch.cuda.synchronize(); score_ms = (time.perf_counter() - t0) * 1000.0
    sample_latency.add_ratio(1.0, score_ms)

    m = hit_metrics(top10, gt_set)
    record(sample_metrics, sample_domain_metrics, 'traditional_sample', m, domain)

print_summary(sample_metrics, sample_domain_metrics, ['traditional_sample'],
              title="Traditional MaxSim — Small Sample")
sample_latency.report()

m = sample_metrics.get('traditional_sample', {'r10':0,'count':1,'n10':0.0})
cnt = max(1, m.get('count', 1))
print(f">>> QUICK RESULT: R@10={m['r10']/cnt*100:.2f}% | nDCG@10={m['n10']/cnt:.4f} | queries={cnt}")
print(">>> Nếu ổn, chạy Cell Method 1 full ngay dưới.")

In [ ]:
# ==============================================================================
# METHOD 1 — Traditional MaxSim (ColPali + ViDoRe)
# Queries encoded live with ColPali (plain forward, no attention capture).
# ==============================================================================

print(">>> METHOD 1: Traditional MaxSim")

trad_metrics        = {}
trad_domain_metrics = {}
trad_query_rows     = []
trad_latency        = LatencyTracker("Traditional MaxSim")

METHOD_KEYS_TRAD = ['traditional']

print(f"Building doc matrix from {len(all_page_embeddings)} page embeddings...")
doc_matrix, doc_mask = build_doc_matrix(all_page_embeddings, device)
n_docs = doc_matrix.shape[0]
print(f"Doc matrix shape: {doc_matrix.shape}")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Traditional MaxSim")

for q_idx, item in pbar:
    question = item['question']
    gt_set   = set(item['gt_embed_indices'])
    domain   = item['domain']

    q_inputs = build_query_inputs(question, query_processor, device)

    with torch.no_grad():
        q_proj = query_model(**q_inputs)

    if hasattr(q_proj, "last_hidden_state") and q_proj.last_hidden_state is not None:
        q_proj = q_proj.last_hidden_state
    elif isinstance(q_proj, (tuple, list)) and len(q_proj) > 0:
        q_proj = q_proj[0]

    attn_mask = q_inputs['attention_mask'][0]
    trad_idx  = torch.where(attn_mask > 0)[0]
    q_emb     = q_proj[0][trad_idx].float()
    q_norm    = F.normalize(q_emb, dim=-1)

    torch.cuda.synchronize()
    t_score_start = time.perf_counter()

    M      = fast_maxsim(q_norm, doc_matrix, doc_mask)
    scores = M.sum(dim=0)
    top10  = torch.topk(scores, min(10, n_docs)).indices.cpu().tolist()

    torch.cuda.synchronize()
    score_ms = (time.perf_counter() - t_score_start) * 1000.0
    trad_latency.add_ratio(1.0, score_ms)

    m = hit_metrics(top10, gt_set)
    record(trad_metrics, trad_domain_metrics, 'traditional', m, domain)

    trad_query_rows.append({
        'query_id':       q_idx,
        'doc_name':       item['doc_name'],
        'domain':         domain,
        'question':       question,
        'trad_r@1':       m['r1'],
        'trad_r@5':       m['r5'],
        'trad_r@10':      m['r10'],
        'trad_ndcg@1':    round(m['n1'],  4),
        'trad_ndcg@5':    round(m['n5'],  4),
        'trad_ndcg@10':   round(m['n10'], 4),
    })

print_summary(trad_metrics, trad_domain_metrics, METHOD_KEYS_TRAD,
              title="Traditional MaxSim Results")
trad_latency.report()

pd.DataFrame(trad_query_rows).to_csv(
    os.path.join(WORKING_DIR, "traditional_queries.csv"), index=False)
print("\n✅ Saved: traditional_queries.csv")

In [ ]:
# ==============================================================================
# METHOD 2 — Ours: SVD Importance + Cluster Pool (ColPali + ViDoRe)
# ==============================================================================
print(">>> METHOD 2: Ours — SVD Importance + Cluster Pool")

ours_metrics = {}; ours_domain_metrics = {}; ours_query_rows = []
ours_latency = LatencyTracker("Ours (SVD+ClusterPool)")

print(f"Running ablation over {len(qa_pairs)} queries")
pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Ours (SVD+ClusterPool)")

for q_idx, item in pbar:
    question = item['question']; gt_set = set(item['gt_embed_indices']); domain = item['domain']
    proj, raw_outputs, q_inputs, encode_ms = encode_query_live(question, query_processor, query_model, device)

    attn_mask_1d = q_inputs['attention_mask'][0].float()
    content_mask_1d = build_content_mask(q_inputs, query_processor)[0].float()
    trad_idx = torch.where(attn_mask_1d > 0)[0]
    method_idx = torch.where(content_mask_1d > 0)[0]
    if trad_idx.numel() == 0: continue
    if method_idx.numel() == 0: method_idx = trad_idx

    content_mask_2d = content_mask_1d.unsqueeze(0)
    embed_cache = {}
    for n_layers in N_LAST_LAYERS_LIST:
        attn_list = []
        if hasattr(raw_outputs,'attentions') and raw_outputs.attentions is not None and len(raw_outputs.attentions) > 0:
            attn_list = list(raw_outputs.attentions[-n_layers:])
        n_actual = len(attn_list)
        layer_weights = torch.exp(torch.linspace(0,1,max(n_actual,1),device=device))
        layer_weights /= layer_weights.sum()
        if n_actual > 0:
            importance = compute_svd_importance_softplus(attn_list, content_mask_2d, layer_weights, SVD_RANK_REMOVE)
        else:
            importance = content_mask_2d.clone()
        importance = (importance * content_mask_2d)[0]
        embed_cache[n_layers] = (proj[0].float(), importance)

    query_row = {'query_id':q_idx,'doc_name':item['doc_name'],'domain':domain,'question':question}

    for n_layers in N_LAST_LAYERS_LIST:
        q_embed, q_importance = embed_cache[n_layers]
        q_method_norm = F.normalize(q_embed[method_idx].float(), dim=-1)
        imp_valid = q_importance[method_idx].float()
        n_method = method_idx.numel()
        sorted_imp_idx = torch.argsort(imp_valid, descending=True)

        for mode in NORMALIZE_MODES:
            for r in TOPK_RATIOS:
                tag = f"L{n_layers}_norm{mode}_r{int(r*100)}"
                n_keep = max(1, int(n_method * r))
                kept_idx = sorted_imp_idx[:n_keep]; disc_idx = sorted_imp_idx[n_keep:]
                q_kept = q_method_norm[kept_idx]; q_disc = q_method_norm[disc_idx]
                imp_kept = imp_valid[kept_idx]; imp_disc = imp_valid[disc_idx]
                pooled_vecs, pooled_weights = build_cluster_pool_vectors_ours(q_kept, q_disc, imp_kept, imp_disc, normalize_mode=mode)
                total_disc_imp = imp_disc.sum().item()
                pool_frac = total_disc_imp / imp_valid.sum().clamp(min=1e-8).item()

                if pooled_vecs.shape[0] > 0:
                    all_q_vecs = torch.cat([q_kept, pooled_vecs], dim=0)
                    all_q_weights = torch.cat([imp_kept, pooled_weights * pool_frac])
                    n_kept_vecs = q_kept.shape[0]
                else:
                    all_q_vecs = q_kept; all_q_weights = imp_kept; n_kept_vecs = q_kept.shape[0]

                torch.cuda.synchronize(); t0 = time.perf_counter()
                M_all = fast_maxsim(all_q_vecs, doc_matrix, doc_mask)
                M_kept_part = M_all[:n_kept_vecs]; M_pooled_part = M_all[n_kept_vecs:]
                sc_trad = M_kept_part.sum(dim=0)
                if M_pooled_part.shape[0] > 0:
                    sc_trad = sc_trad + (M_pooled_part * pooled_weights.unsqueeze(-1)).sum(dim=0) / max(total_disc_imp, 1e-8) * pool_frac
                top10_trad = torch.topk(sc_trad, min(10, n_docs)).indices.cpu().tolist()
                all_w_n = all_q_weights / all_q_weights.sum().clamp(min=1e-8)
                sc_wt = (M_all * all_w_n.unsqueeze(-1)).sum(dim=0)
                top10_wt = torch.topk(sc_wt, min(10, n_docs)).indices.cpu().tolist()
                torch.cuda.synchronize(); score_ms = (time.perf_counter()-t0)*1000.0
                ours_latency.add_ratio(r, score_ms)

                m_trad = hit_metrics(top10_trad, gt_set); key_trad = f"{tag}_trad"
                record(ours_metrics, ours_domain_metrics, key_trad, m_trad, domain)
                query_row.update({f'{key_trad}_r@10':m_trad['r10'],f'{key_trad}_ndcg@10':round(m_trad['n10'],4)})

                m_wt = hit_metrics(top10_wt, gt_set); key_wt = f"{tag}_weighted"
                record(ours_metrics, ours_domain_metrics, key_wt, m_wt, domain)
                query_row.update({f'{key_wt}_r@10':m_wt['r10'],f'{key_wt}_ndcg@10':round(m_wt['n10'],4)})

    ours_query_rows.append(query_row)
    if q_idx % 50 == 0: gc.collect(); torch.cuda.empty_cache()

print_summary(ours_metrics, ours_domain_metrics, ABLATION_KEYS_OURS, title="Ours — SVD Importance + Cluster Pool")
ours_latency.report()
pd.DataFrame(ours_query_rows).to_csv(os.path.join(WORKING_DIR,"ours_ablation_queries.csv"),index=False)
print("\n✅ Saved: ours_ablation_queries.csv")


In [ ]:
# ==============================================================================
# METHOD 3 — Hierarchical Ward Token Pooling (ColPali + ViDoRe)
# ==============================================================================
print(">>> METHOD 3: Hierarchical Ward Token Pooling")

def _ward_distance_matrix(cents, sizes):
    sim = torch.matmul(cents,cents.t()).clamp(-1.0,1.0)
    sq_dist = 2.0*(1.0-sim)
    ni = sizes.float().unsqueeze(1); nj = sizes.float().unsqueeze(0)
    ward = (ni*nj)/(ni+nj)*sq_dist
    mask = torch.ones(cents.shape[0],cents.shape[0],dtype=torch.bool,device=cents.device).tril()
    ward.masked_fill_(mask,float('inf'))
    return ward

def ward_pool(vecs, n_clusters):
    N = vecs.shape[0]
    if N <= n_clusters: return vecs.clone()
    dev = vecs.device; sums = vecs.clone().float()
    sizes = torch.ones(N,device=dev,dtype=torch.float32)
    cents = F.normalize(sums,dim=-1); active = torch.ones(N,dtype=torch.bool,device=dev); C = N
    while C > n_clusters:
        active_idx = active.nonzero(as_tuple=True)[0]
        ward = _ward_distance_matrix(cents[active_idx],sizes[active_idx])
        flat_idx = ward.argmin().item(); n_act = active_idx.shape[0]
        ai, aj = flat_idx//n_act, flat_idx%n_act
        gi, gj = active_idx[ai].item(), active_idx[aj].item()
        sums[gi] = sums[gi]+sums[gj]; sizes[gi] = sizes[gi]+sizes[gj]
        cents[gi] = F.normalize(sums[gi].unsqueeze(0),dim=-1).squeeze(0)
        active[gj] = False; C -= 1
    return cents[active.nonzero(as_tuple=True)[0]]

def ward_pool_scores_all_ratios(q_norm, doc_matrix, doc_mask, topk_ratios):
    N = q_norm.shape[0]; results = {}; timing = {}
    prev_vecs, prev_C = q_norm.clone(), N
    for ratio in sorted(topk_ratios, reverse=True):
        target_C = max(1, round(N*ratio))
        if target_C >= prev_C: centroids = prev_vecs
        else:
            centroids = ward_pool(prev_vecs, target_C); prev_vecs, prev_C = centroids, centroids.shape[0]
        torch.cuda.synchronize(); t0 = time.perf_counter()
        M_c = fast_maxsim(centroids,doc_matrix,doc_mask); results[ratio] = M_c.sum(0)
        torch.cuda.synchronize(); timing[ratio] = (time.perf_counter()-t0)*1000.0
    return results, timing

METHOD_KEYS_HIER = ['traditional'] + [f"hier_r{int(r*100)}" for r in TOPK_RATIOS]
hier_metrics = {}; hier_domain_metrics = {}; hier_query_rows = []; hier_latency = LatencyTracker("Hierarchical Ward Pool")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Hierarchical Ward Pool")
for q_idx, item in pbar:
    question = item['question']; gt_set = set(item['gt_embed_indices']); domain = item['domain']
    q_inputs = build_query_inputs(question, query_processor, device)
    q_norm = forward_query_embeddings(query_model, q_inputs)

    M_trad = fast_maxsim(q_norm,doc_matrix,doc_mask); trad_sc = M_trad.sum(dim=0)
    top10_trad = torch.topk(trad_sc,min(10,n_docs)).indices.cpu().tolist()
    m_trad = hit_metrics(top10_trad,gt_set)
    record(hier_metrics,hier_domain_metrics,'traditional',m_trad,domain)

    ratio_scores, ratio_timing = ward_pool_scores_all_ratios(q_norm,doc_matrix,doc_mask,TOPK_RATIOS)
    query_row = {'query_id':q_idx,'doc_name':item['doc_name'],'domain':domain,'question':question,
                 'trad_r@1':m_trad['r1'],'trad_r@5':m_trad['r5'],'trad_r@10':m_trad['r10'],'trad_ndcg@10':round(m_trad['n10'],4)}
    for r in TOPK_RATIOS:
        key = f"hier_r{int(r*100)}"
        top10 = torch.topk(ratio_scores[r],min(10,n_docs)).indices.cpu().tolist()
        m = hit_metrics(top10,gt_set)
        record(hier_metrics,hier_domain_metrics,key,m,domain); hier_latency.add_ratio(r,ratio_timing[r])
        query_row.update({f'{key}_r@10':m['r10'],f'{key}_ndcg@10':round(m['n10'],4)})
    hier_query_rows.append(query_row)

print_summary(hier_metrics,hier_domain_metrics,METHOD_KEYS_HIER,title="Hierarchical Ward Pooling Results")
hier_latency.report()
pd.DataFrame(hier_query_rows).to_csv(os.path.join(WORKING_DIR,"hierarchical_queries.csv"),index=False)
print("\n✅ Saved: hierarchical_queries.csv")


In [ ]:
# ==============================================================================
# METHOD 4 — Attention Score Token Pruning (ColPali + ViDoRe)
# ==============================================================================
print(">>> METHOD 4: Attention Score Token Pruning")

METHOD_KEYS_ATTN = ['traditional']
for n in ATTN_N_LAYERS_LIST:
    for r in TOPK_RATIOS:
        METHOD_KEYS_ATTN += [f"attn_L{n}_r{int(r*100)}_trad", f"attn_L{n}_r{int(r*100)}_weighted"]

attn_metrics = {}; attn_domain_metrics = {}; attn_query_rows = []; attn_latency = LatencyTracker("Attention Score Pruning")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Attention Score Pruning")
for q_idx, item in pbar:
    question = item['question']; gt_set = set(item['gt_embed_indices']); domain = item['domain']
    proj, raw_outputs, q_inputs, encode_ms = encode_query_live(question, query_processor, query_model, device)

    attn_mask_1d = q_inputs['attention_mask'][0].float()
    trad_idx = torch.where(attn_mask_1d > 0)[0]; N = trad_idx.numel()
    q_norm = F.normalize(proj[0][trad_idx].float(), dim=-1)

    M_full = fast_maxsim(q_norm,doc_matrix,doc_mask); trad_sc = M_full.sum(dim=0)
    top10_trad = torch.topk(trad_sc,min(10,n_docs)).indices.cpu().tolist()
    m_trad = hit_metrics(top10_trad,gt_set)
    record(attn_metrics,attn_domain_metrics,'traditional',m_trad,domain)

    query_row = {'query_id':q_idx,'doc_name':item['doc_name'],'domain':domain,'question':question,
                 'trad_r@1':m_trad['r1'],'trad_r@5':m_trad['r5'],'trad_r@10':m_trad['r10'],'trad_ndcg@10':round(m_trad['n10'],4)}

    for n_layers in ATTN_N_LAYERS_LIST:
        if hasattr(raw_outputs,'attentions') and raw_outputs.attentions is not None and len(raw_outputs.attentions) >= 1:
            attn_subset = list(raw_outputs.attentions[-n_layers:])
            imp_2d = torch.zeros(1, q_inputs['attention_mask'].shape[1], device=device)
            for attn_layer in attn_subset:
                imp_2d += attn_layer.float().sum(dim=2).mean(dim=1)
            imp = imp_2d[0][trad_idx]
        else:
            imp = torch.ones(N, device=device)

        sorted_imp_idx = torch.argsort(imp, descending=True)
        for r in TOPK_RATIOS:
            n_keep = max(1, int(N*r))
            kept_idx = sorted_imp_idx[:n_keep]; imp_kept = imp[kept_idx]; q_kept = q_norm[kept_idx]

            torch.cuda.synchronize(); t0 = time.perf_counter()
            M_kept = fast_maxsim(q_kept,doc_matrix,doc_mask)
            sc_trad = M_kept.sum(dim=0); top_t = torch.topk(sc_trad,min(10,n_docs)).indices.cpu().tolist()
            imp_k_n = imp_kept/imp_kept.sum().clamp(min=1e-8)
            sc_w = (M_kept*imp_k_n.unsqueeze(-1)).sum(dim=0); top_w = torch.topk(sc_w,min(10,n_docs)).indices.cpu().tolist()
            torch.cuda.synchronize(); attn_latency.add_ratio(r,(time.perf_counter()-t0)*1000.0)

            m_t = hit_metrics(top_t,gt_set); key_t = f"attn_L{n_layers}_r{int(r*100)}_trad"
            record(attn_metrics,attn_domain_metrics,key_t,m_t,domain)
            query_row.update({f'{key_t}_r@10':m_t['r10'],f'{key_t}_ndcg@10':round(m_t['n10'],4)})

            m_w = hit_metrics(top_w,gt_set); key_w = f"attn_L{n_layers}_r{int(r*100)}_weighted"
            record(attn_metrics,attn_domain_metrics,key_w,m_w,domain)
            query_row.update({f'{key_w}_r@10':m_w['r10'],f'{key_w}_ndcg@10':round(m_w['n10'],4)})

    attn_query_rows.append(query_row)

print_summary(attn_metrics,attn_domain_metrics,['traditional']+[f"attn_L1_r{int(r*100)}_trad" for r in TOPK_RATIOS],
              title="Attention Score Pruning (L=1, trad)")
attn_latency.report()
pd.DataFrame(attn_query_rows).to_csv(os.path.join(WORKING_DIR,"attention_queries.csv"),index=False)
print("\n✅ Saved: attention_queries.csv")


In [ ]:
# ==============================================================================
# METHOD 5 — Spherical KMeans Token Pooling (ColPali + ViDoRe)
# ==============================================================================
print(">>> METHOD 5: Spherical KMeans Token Pooling")

METHOD_KEYS_KMEANS = ['traditional'] + [f"kmeans_r{int(r*100)}" for r in TOPK_RATIOS]
kmeans_metrics = {}; kmeans_domain_metrics = {}; kmeans_query_rows = []; kmeans_latency = LatencyTracker("Spherical KMeans Pool")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Spherical KMeans Pool")
for q_idx, item in pbar:
    question = item['question']; gt_set = set(item['gt_embed_indices']); domain = item['domain']
    q_inputs = build_query_inputs(question, query_processor, device)
    q_norm = forward_query_embeddings(query_model, q_inputs); N = q_norm.shape[0]

    M_full = fast_maxsim(q_norm,doc_matrix,doc_mask); trad_sc = M_full.sum(dim=0)
    top10_trad = torch.topk(trad_sc,min(10,n_docs)).indices.cpu().tolist()
    m_trad = hit_metrics(top10_trad,gt_set)
    record(kmeans_metrics,kmeans_domain_metrics,'traditional',m_trad,domain)

    query_row = {'query_id':q_idx,'doc_name':item['doc_name'],'domain':domain,'question':question,
                 'trad_r@1':m_trad['r1'],'trad_r@5':m_trad['r5'],'trad_r@10':m_trad['r10'],'trad_ndcg@10':round(m_trad['n10'],4)}

    for r in TOPK_RATIOS:
        K = max(1, int(N*r))
        centroids = spherical_kmeans(q_norm, K)
        torch.cuda.synchronize(); t0 = time.perf_counter()
        M_k = fast_maxsim(centroids,doc_matrix,doc_mask); scores = M_k.sum(dim=0)
        top10 = torch.topk(scores,min(10,n_docs)).indices.cpu().tolist()
        torch.cuda.synchronize(); kmeans_latency.add_ratio(r,(time.perf_counter()-t0)*1000.0)

        key = f"kmeans_r{int(r*100)}"; m = hit_metrics(top10,gt_set)
        record(kmeans_metrics,kmeans_domain_metrics,key,m,domain)
        query_row.update({f'{key}_r@10':m['r10'],f'{key}_ndcg@10':round(m['n10'],4)})
    kmeans_query_rows.append(query_row)

print_summary(kmeans_metrics,kmeans_domain_metrics,METHOD_KEYS_KMEANS,title="Spherical KMeans Pooling Results")
kmeans_latency.report()
pd.DataFrame(kmeans_query_rows).to_csv(os.path.join(WORKING_DIR,"kmeans_queries.csv"),index=False)
print("\n✅ Saved: kmeans_queries.csv")


In [ ]:
# ==============================================================================
# METHOD 6 — Random Token Pruning (ColPali + ViDoRe)
# ==============================================================================
print(">>> METHOD 6: Random Token Pruning (Ablation Baseline)")

METHOD_KEYS_RAND = ['traditional'] + [f"rand_r{int(r*100)}" for r in TOPK_RATIOS]
rand_metrics = {}; rand_domain_metrics = {}; rand_query_rows = []; rand_latency = LatencyTracker("Random Pruning")

pbar = tqdm(enumerate(qa_pairs), total=len(qa_pairs), desc="Random Pruning")
for q_idx, item in pbar:
    question = item['question']; gt_set = set(item['gt_embed_indices']); domain = item['domain']
    q_inputs = build_query_inputs(question, query_processor, device)
    q_norm = forward_query_embeddings(query_model, q_inputs)

    M_full = fast_maxsim(q_norm,doc_matrix,doc_mask); trad_sc = M_full.sum(dim=0)
    top10_trad = torch.topk(trad_sc,min(10,n_docs)).indices.cpu().tolist()
    m_trad = hit_metrics(top10_trad,gt_set)
    record(rand_metrics,rand_domain_metrics,'traditional',m_trad,domain)

    query_row = {'query_id':q_idx,'doc_name':item['doc_name'],'domain':domain,'question':question,
                 'trad_r@1':m_trad['r1'],'trad_r@5':m_trad['r5'],'trad_r@10':m_trad['r10'],
                 'trad_ndcg@10':round(m_trad['n10'],4),'n_random_seeds':N_RANDOM_SEEDS}

    ratio_scores, ratio_timing = random_prune_topk(q_norm, None, doc_matrix, doc_mask, TOPK_RATIOS, N_RANDOM_SEEDS)
    for r in TOPK_RATIOS:
        key = f"rand_r{int(r*100)}"; rand_latency.add_ratio(r, ratio_timing[r])
        top10 = torch.topk(ratio_scores[r],min(10,n_docs)).indices.cpu().tolist()
        m = hit_metrics(top10,gt_set)
        record(rand_metrics,rand_domain_metrics,key,m,domain)
        query_row.update({f'{key}_r@10':m['r10'],f'{key}_ndcg@10':round(m['n10'],4)})
    rand_query_rows.append(query_row)

print_summary(rand_metrics,rand_domain_metrics,METHOD_KEYS_RAND,title="Random Token Pruning Results")
rand_latency.report()
pd.DataFrame(rand_query_rows).to_csv(os.path.join(WORKING_DIR,"random_pruning_queries.csv"),index=False)
print("\n✅ Saved: random_pruning_queries.csv")


In [ ]:
# ==============================================================================
# FINAL SUMMARY — aggregate results from all methods
# ==============================================================================
print("="*80)
print("FINAL SUMMARY — ColPali v1.3 × ViDoRe V3")
print("="*80)

print_summary(trad_metrics, trad_domain_metrics, ['traditional'], title="Traditional MaxSim")
print_summary(hier_metrics, hier_domain_metrics, METHOD_KEYS_HIER, title="Hierarchical Ward Pooling")
print_summary(ours_metrics, ours_domain_metrics, ABLATION_KEYS_OURS[:10] + ['...'], title="Ours — SVD+ClusterPool (first 10 keys)")
print_summary(attn_metrics, attn_domain_metrics,
              ['traditional'] + [f"attn_L1_r{int(r*100)}_trad" for r in TOPK_RATIOS],
              title="Attention Score Pruning (L=1)")
print_summary(kmeans_metrics, kmeans_domain_metrics, METHOD_KEYS_KMEANS, title="Spherical KMeans Pooling")
print_summary(rand_metrics, rand_domain_metrics, METHOD_KEYS_RAND, title="Random Token Pruning")

# Save master CSVs
save_summary_csv(trad_metrics, trad_domain_metrics, ['traditional'], "traditional")
save_summary_csv(hier_metrics, hier_domain_metrics, METHOD_KEYS_HIER, "hierarchical")
save_summary_csv(ours_metrics, ours_domain_metrics, ABLATION_KEYS_OURS, "ours_ablation")
save_summary_csv(attn_metrics, attn_domain_metrics, METHOD_KEYS_ATTN, "attention_pruning")
save_summary_csv(kmeans_metrics, kmeans_domain_metrics, METHOD_KEYS_KMEANS, "spherical_kmeans")
save_summary_csv(rand_metrics, rand_domain_metrics, METHOD_KEYS_RAND, "random_pruning")

# Save latency reports
latency_rows = []
for tracker in [trad_latency, ours_latency, hier_latency, attn_latency, kmeans_latency, rand_latency]:
    latency_rows.extend(tracker.to_dict())
if latency_rows:
    pd.DataFrame(latency_rows).to_csv(os.path.join(WORKING_DIR, "MASTER_latency.csv"), index=False)
    print("✅ Saved: MASTER_latency.csv")

print("\n>>> All evaluations complete.")
